# FINE-TUNNING FOR OPENVLA OFT

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
# FINE-TUNNING SCRIPT FOR OPENVLA OFT MODEL PART 1
# CONFIGURATIONS AND FUNCTIONS

import os
os.chdir("/home/ids/ext-5219/tokenizer/openvla-oft/")  # replace with your repo root
print("Current working directory:", os.getcwd())

sys.argv.append("pusht")


import time
from torch.nn.parallel import DistributedDataParallel as DDP
from typing import Dict, Optional, Tuple, Type
from pathlib import Path
from transformers.modeling_outputs import CausalLMOutputWithPast
import torch.nn as nn
import torch
from prismatic.training.train_utils import (
    compute_actions_l1_loss,
    compute_token_accuracy,
    get_current_action_mask,
    get_next_actions_mask,
)
from prismatic.vla.constants import (
    ACTION_DIM,
    ACTION_PROPRIO_NORMALIZATION_TYPE,
    NUM_ACTIONS_CHUNK,
    PROPRIO_DIM,
)
from prismatic.vla.datasets.rlds.utils.data_utils import save_dataset_statistics
import torch.distributed as dist
from transformers import AutoConfig, AutoImageProcessor, AutoModelForVision2Seq, AutoProcessor
from peft import LoraConfig, PeftModel, get_peft_model
import wandb


# Configurations
vla_path: str = "openvla/openvla-7b"             # Path to OpenVLA model (on HuggingFace Hub or stored locally)
run_root_dir: Path = Path("runs")   
# Trim trailing forward slash ('/') in VLA path if it exists
# vla_path = vla_path.rstrip("/")
print(f"Fine-tuning OpenVLA Model `{vla_path}` ")
num_images_in_input= 1 #number of images in input
use_lora=True
use_film=False
lora_rank: int = 32                              # Rank of LoRA weight matrix
lora_dropout: float = 0.0     
resume=False 
resume_step: Optional[int] = None                # (When `resume==True`) Step number that we are resuming from
use_proprio=False
use_l1_regression=False
use_diffusion=False
use_subtrajectory=True
# continuous action head with diffusion modeling objective (DDIM)
num_diffusion_steps_train: int = 50              # (When `diffusion==True`) 
learning_rate: float = 5e-4                      # Learning rate
num_steps_before_decay: int = 100_000            # Number of steps before LR decays by 10x
  # Dataset
# data_root_dir: Path = Path("tf_datasets") 
# data_root_dir: Path = Path("/tsi/hi-paris/Pollen/datasets/tf_datasets")
data_root_dir: Path = Path("/home/ids/ext-5219/tokenizer/test")

dataset_name: str = "columbia_cairlab_pusht_real"   # Name of fine-tuning dataset (e.g., `aloha_scoop_x_into_bowl`)
shuffle_buffer_size: int = 1000               # Dataloader shuffle buffer size (can reduce if OOM errors occur)
image_aug: bool = True                           # If True, trains with image augmentations (HIGHLY RECOMMENDED)
use_val_set: bool = False                        # If True, uses validation set and log validation metrics
batch_size: int = 8    
grad_accumulation_steps: int = 1                 # Number of gradient accumulation steps
# max_steps: int = 200_000                         # Max number of training steps
max_steps: int = 300 
diffusion_sample_freq: int = 50                  # (When `use_diffusion==True`) Frequency for sampling in steps
lr_warmup_steps: int = 0                         # Number of steps to warm up learning rate (from 10% to 100%)
save_freq: int = 150                          # Checkpoint saving frequency in steps
val_freq: int = 10_000                           # (When `use_val_set==True`) Validation set logging frequency in steps
val_time_limit: int = 180                        # (When `use_val_set==True`) Time limit for computing validation metrics
save_latest_checkpoint_only: bool = True        # If True, saves only 1 checkpoint, overwriting latest checkpoint
merge_lora_during_training: bool = True          # If True, merges LoRA weights and saves result during training
SUBTRAJECTORY_DIM=30                       # Dimension of subtrajectory ID embeddings
def log_metrics_to_wandb(metrics, prefix, step, wandb_entity) -> None:
    """
    Log metrics to Weights & Biases.

    Args:
        metrics (dict): Dictionary of metrics to log
        prefix (str): Prefix for metric names
        step (int): Training step
        wandb_entity (str): W&B entity instance

    Returns:
        None.
    """
    log_dict = {}
    for name, value in metrics.items():
        # Map loss_value to Loss for better readability in W&B
        if name == "loss_value":
            log_dict[f"{prefix}/Loss"] = value
        # Keep other metrics as is
        else:
            log_dict[f"{prefix}/{name.replace('_', ' ').title()}"] = value
    wandb_entity.log(log_dict, step=step)



def compute_smoothened_metrics(metrics_deques) -> dict:
    """
    Compute smoothened metrics from recent deques.

    Args:
        metrics_deques (dict): Dictionary of deques containing recent metrics.

    Returns:
        dict: Dictionary of smoothened metrics.
    """
    smoothened_metrics = {}
    for name, deque in metrics_deques.items():
        if deque and len(deque) > 0:
            smoothened_metrics[name] = sum(deque) / len(deque)
    return smoothened_metrics

def load_checkpoint(module_name: str, path: str, step: int, device: str = "cpu") -> dict:
    """
    Loads a checkpoint for a given module.

    Args:
        module_name (str): Name of model component to load checkpoint for.
        path (str): Path to checkpoint directory.
        step (int): Gradient step number of saved checkpoint.
        device (str): String specifying how to remap storage locations (default = "cpu").

    Returns:
        dict: PyTorch model state dictionary.
    """
    checkpoint_path = os.path.join(path, f"{module_name}--{step}_checkpoint.pt")
    print(f"Loading checkpoint: {checkpoint_path}")
    state_dict = torch.load(checkpoint_path, weights_only=True, map_location=device)
    return remove_ddp_in_checkpoint(state_dict)

def remove_ddp_in_checkpoint(state_dict) -> dict:
    """
    Removes the 'module.' prefix from parameter names in a PyTorch model state dictionary that was saved using
    DistributedDataParallel (DDP).

    When a model is trained using PyTorch's DistributedDataParallel, the saved state dictionary contains parameters
    prefixed with 'module.'. This function removes these prefixes to make the state dictionary compatible when
    loading into models that are not yet wrapped in DDP.

    Args:
        state_dict (dict): PyTorch model state dictionary.

    Returns:
        dict: A new state dictionary with the same contents but with 'module.' prefixes removed from parameter names.
              Parameters without the 'module.' prefix remain unchanged.
    """
    new_state_dict = {}
    for k, v in state_dict.items():
        if k[:7] == "module.":
            new_state_dict[k[7:]] = v
        else:
            new_state_dict[k] = v
    return new_state_dict

def wrap_ddp(module: nn.Module, device_id: int, find_unused: bool = False) -> DDP:
    """
    Wrap a module with DistributedDataParallel.

    Args:
        module (nn.Module): PyTorch module.
        device_id (str): Device ID.
        find_unused (bool): Whether to detect parameters without gradients in distributed training.

    Returns:
        DistributedDataParallel: PyTorch module wrapped with DDP.
    """
    return DDP(module, device_ids=[device_id], find_unused_parameters=find_unused, gradient_as_bucket_view=True)

def init_module(
    module_class: Type[nn.Module],
    module_name: str,
    device_id: int,
    module_args: dict,
    to_bf16: bool = False,
    find_unused_params: bool = False,
) -> DDP:
    """
    Initializes a module, optionally loads checkpoint, moves to device, and wraps with DDP.

    Args:
        module_class (Type[nn.Module]): Class of PyTorch module to initialize.
        module_name (str): Name of model component to load checkpoint for.
        cfg (FinetuneConfig): Training configuration.
        device_id (str): Device ID.
        module_args (dict): Args for initializing the module.
        to_bf16 (bool): Whether to convert to torch.bfloat16 data type.
        find_unused_params (bool): Whether to detect parameters without gradients in distributed training.

    Returns:
        DistributedDataParallel: PyTorch module wrapped with DDP.
    """
    module = module_class(**module_args)


    if resume:
        state_dict = load_checkpoint(module_name, vla_path, resume_step)
        module.load_state_dict(state_dict)

    if to_bf16:
        module = module.to(torch.bfloat16)
    module = module.to(device_id)

    return wrap_ddp(module, device_id, find_unused_params)

def run_diffusion_sampling(
    vla,
    action_head,
    noisy_action_projector,
    proprio_projector,
    batch,
    batch_size,
    num_patches,
    actions_shape,
    device_id,
    current_action_mask,
    next_actions_mask,
    use_proprio,
    use_film,
) -> torch.Tensor:
    """
    Run diffusion sampling (reverse diffusion) to generate actions.

    Args:
        vla (OpenVLAForActionPrediction): Vision-language-action policy.
        action_head (nn.Module): Action head module.
        noisy_action_projector (nn.Module): Noisy action projector module (only used for diffusion).
        proprio_projector (nn.Module): Proprioceptive state projector module.
        batch (dict): Input batch.
        batch_size (int): Batch size.
        num_patches (int): Number of vision patches.
        actions_shape (tuple): Shape of ground-truth actions.
        device_id (str): Device ID.
        current_action_mask (torch.Tensor): Mask for current action.
        next_actions_mask (torch.Tensor): Mask for next actions.
        use_proprio (bool): Whether to use proprioceptive state as input.
        use_film (bool): Whether to use FiLM for better language following.

    Returns:
        torch.Tensor: Predicted actions.
    """
    # Sample random noisy action, used as the starting point for reverse diffusion
    noise = torch.randn(
        size=(batch_size, NUM_ACTIONS_CHUNK, ACTION_DIM),
        device=device_id,
        dtype=torch.bfloat16,
    )  # (B, chunk_len, action_dim)

    # Set diffusion timestep values
    action_head.module.noise_scheduler.set_timesteps(action_head.module.num_diffusion_steps_train)

    # Reverse diffusion: Iteratively denoise to generate action, conditioned on observation
    curr_noisy_actions = noise
    for t in action_head.module.noise_scheduler.timesteps:
        # Get diffusion model's noise prediction (conditioned on VLA latent embedding, current noisy action embedding,
        # and diffusion timestep embedding)
        timesteps = torch.Tensor([t]).repeat(batch_size).to(device_id)
        diffusion_timestep_embeddings = (
            action_head.module.time_encoder(timesteps).to(curr_noisy_actions.dtype).to(curr_noisy_actions.device)
        )  # (B, llm_dim)
        diffusion_timestep_embeddings = diffusion_timestep_embeddings.unsqueeze(1)  # (B, 1, llm_dim)

        with torch.autocast("cuda", dtype=torch.bfloat16):
            output = vla(
                input_ids=batch["input_ids"].to(device_id),
                attention_mask=batch["attention_mask"].to(device_id),
                pixel_values=batch["pixel_values"].to(torch.bfloat16).to(device_id),
                labels=batch["labels"].to(device_id),
                output_hidden_states=True,
                proprio=batch["proprio"] if use_proprio else None,
                proprio_projector=proprio_projector if use_proprio else None,
                noisy_actions=curr_noisy_actions,
                noisy_action_projector=noisy_action_projector,
                diffusion_timestep_embeddings=diffusion_timestep_embeddings,
                use_film=use_film,
            )
            # Get last layer hidden states
            last_hidden_states = output.hidden_states[-1]  # (B, seq_len, D)
            # Get hidden states for text portion of prompt+response (after the vision patches)
            text_hidden_states = last_hidden_states[:, num_patches:-1]
            # Get hidden states for action portion of response
            actions_hidden_states = text_hidden_states[current_action_mask | next_actions_mask].reshape(
                batch_size, NUM_ACTIONS_CHUNK * ACTION_DIM, -1
            )  # (B, act_chunk_len, D)
            actions_hidden_states = actions_hidden_states.to(torch.bfloat16)
            # Predict noise
            noise_pred = action_head.module.predict_noise(actions_hidden_states)

        # Compute the action at the previous diffusion timestep: x_t -> x_{t-1}
        curr_noisy_actions = action_head.module.noise_scheduler.step(noise_pred, t, curr_noisy_actions).prev_sample

    return curr_noisy_actions.reshape(actions_shape)

def run_forward_pass(
    vla,
    action_head,
    noisy_action_projector,
    proprio_projector,
    batch,
    action_tokenizer,
    device_id,
    use_l1_regression,
    use_diffusion,
    use_proprio,
    use_film,
    num_patches,
    compute_diffusion_l1=False,
    num_diffusion_steps_train=None,
    use_subtrajectory=False,
) -> Tuple[torch.Tensor, Dict[str, float]]:
    """
    Compute model forward pass and metrics for both training and validation.

    Args:
        vla (OpenVLAForActionPrediction): Vision-language-action policy.
        action_head (nn.Module): Action head module.
        noisy_action_projector (nn.Module): Noisy action projector module (only used for diffusion).
        proprio_projector (nn.Module): Proprioceptive state projector module.
        batch (dict): Input batch.
        action_tokenizer (ActionTokenizer): Action tokenizer.
        device_id (str): Device ID.
        use_l1_regression (bool): Whether to use L1 regression.
        use_diffusion (bool): Whether to use diffusion.
        use_proprio (bool): Whether to use proprioceptive state as input.
        use_film (bool): Whether to use FiLM for better language following.
        num_patches (int): Number of vision patches.
        compute_diffusion_l1 (bool): Whether to sample actions and compute L1 loss for diffusion (do this once every
                                    diffusion_sample_freq steps during training; do it every batch for validation)
        num_diffusion_steps_train (int): Number of diffusion steps for training (only used for diffusion).

    Returns:
        tuple: (loss, metrics_dict)
            loss: The loss tensor with gradient for backpropagation.
            metrics_dict: Dictionary of computed metrics (detached values for logging).
    """
    metrics = {}

    # Get ground-truth action labels
    ground_truth_actions = batch["actions"].to(device_id).to(torch.bfloat16)
    print(f"ground_truth_actions shape: {ground_truth_actions.shape}")



    # Main keys: dict_keys(['pixel_values', 'proprio', 'input_ids', 'attention_mask', 'labels', 'actions', 'dataset_names'])

    # [Only for diffusion] Sample noisy actions used as input for noise predictor network
    if use_diffusion:
        noisy_dict = action_head.module.sample_noisy_actions(ground_truth_actions)
        noise, noisy_actions, diffusion_timestep_embeddings = (
            noisy_dict["noise"],
            noisy_dict["noisy_actions"],
            noisy_dict["diffusion_timestep_embeddings"],
        )
    else:
        noise, noisy_actions, diffusion_timestep_embeddings = None, None, None

    # VLA forward pass
    with torch.autocast("cuda", dtype=torch.bfloat16):
        output: CausalLMOutputWithPast = vla(
            input_ids=batch["input_ids"].to(device_id),
            attention_mask=batch["attention_mask"].to(device_id),
            pixel_values=batch["pixel_values"].to(torch.bfloat16).to(device_id),
            labels=batch["labels"].to(device_id),
            output_hidden_states=True,
            proprio=batch["proprio"] if use_proprio else None,
            proprio_projector=proprio_projector if use_proprio else None,
            noisy_actions=noisy_actions if use_diffusion else None,
            noisy_action_projector=noisy_action_projector if use_diffusion else None,
            diffusion_timestep_embeddings=diffusion_timestep_embeddings if use_diffusion else None,
            use_film=use_film,
        )

    # Get action masks needed for logging
    ground_truth_token_ids = batch["labels"][:, 1:].to(device_id)
    current_action_mask = get_current_action_mask(ground_truth_token_ids)
    next_actions_mask = get_next_actions_mask(ground_truth_token_ids)

    # Compute metrics for discrete action representation (next-token prediction)
    if not (use_l1_regression or use_diffusion or use_subtrajectory):
        loss = output.loss
        predicted_token_ids = output.logits[:, num_patches:-1].argmax(dim=2)
        curr_action_accuracy = compute_token_accuracy(
            predicted_token_ids, ground_truth_token_ids, mask=current_action_mask
        )
        curr_action_l1_loss = compute_actions_l1_loss(
            action_tokenizer, predicted_token_ids, ground_truth_token_ids, mask=current_action_mask
        )
        next_actions_accuracy = compute_token_accuracy(
            predicted_token_ids, ground_truth_token_ids, mask=next_actions_mask
        )
        next_actions_l1_loss = compute_actions_l1_loss(
            action_tokenizer, predicted_token_ids, ground_truth_token_ids, mask=next_actions_mask
        )
        metrics.update(
            {
                "loss_value": loss.item(),  # Detached value for logging
                "curr_action_accuracy": curr_action_accuracy.item(),
                "curr_action_l1_loss": curr_action_l1_loss.item(),
                "next_actions_accuracy": next_actions_accuracy.item(),
                "next_actions_l1_loss": next_actions_l1_loss.item(),
            }
        )
    # Compute metrics for continuous action representations (L1 regression | diffusion)
    else:

# last_hidden_states: torch.Size([8, 370, 4096]) B= 8, seq_len=370, D=4096
# text_hidden_states: torch.Size([8, 113, 4096]) 113=num of patches
#  batch input_ids: torch.Size([8, 114])
#  actions_hidden_states [8 240 -1] (B, act_chunk_len, D)
        # Get last layer hidden states
        last_hidden_states = output.hidden_states[-1]  # (B, seq_len, D)
        print(f"last_hidden_states: {last_hidden_states.shape}")

        # Get hidden states for text portion of prompt+response (after the vision patches)
        text_hidden_states = last_hidden_states[:, num_patches:-1]
        print(f"text_hidden_states: {text_hidden_states.shape}")

        # Get hidden states for action portion of response
        batch_size = batch["input_ids"].shape[0]
        # n=batch["input_ids"]
        # print(f" batch input_ids: { n.shape}")

        actions_hidden_states = (
            text_hidden_states[current_action_mask | next_actions_mask]
            .reshape(batch_size, NUM_ACTIONS_CHUNK * ACTION_DIM, -1)
            .to(torch.bfloat16)
        )  # (B, act_chunk_len, D) 8 240 -1
        # print(f"actions_hidden_states: {actions_hidden_states.shape}")
        if use_l1_regression:
            # Predict action
          
            predicted_actions = action_head.module.predict_action(actions_hidden_states)
            # Get full L1 loss
            loss = torch.nn.L1Loss()(ground_truth_actions, predicted_actions)

        if use_diffusion:
            # Predict noise
            noise_pred = action_head.module.predict_noise(actions_hidden_states)
            # Get diffusion noise prediction MSE loss
            noise_pred = noise_pred.reshape(noise.shape)
            loss = nn.functional.mse_loss(noise_pred, noise, reduction="mean")

            # Only sample actions and compute L1 losses if specified
            if compute_diffusion_l1:
                with torch.no_grad():
                    predicted_actions = run_diffusion_sampling(
                        vla=vla,
                        action_head=action_head,
                        noisy_action_projector=noisy_action_projector,
                        proprio_projector=proprio_projector,
                        batch=batch,
                        batch_size=batch_size,
                        num_patches=num_patches,
                        actions_shape=ground_truth_actions.shape,
                        device_id=device_id,
                        current_action_mask=current_action_mask,
                        next_actions_mask=next_actions_mask,
                        use_proprio=use_proprio,
                        use_film=use_film,
                    )
        
        if use_subtrajectory:
            # print("HELLO 1")
               # Predict action
            # Get ground-truth subtrajectory id label
            ground_truth_subtrajectory_id = batch["subtrajectory_id"].to(device_id)
               # ground_truth_actions: ground-truth actions
                # - shape: (batch_size, chunk_len, action_dim)=> prediction
                # - shape: (batch_size, chunk_len) =>ground truth
            head = action_head.module if hasattr(action_head, "module") else action_head
            predicted_actions = head.predict_action(actions_hidden_states)
            # print("HELLO 2")
            # Get full Cross Entropy loss
            # loss = torch.nn.CrossEntropyLoss()(predicted_actions,ground_truth_subtrajectory_id)

            logits = predicted_actions.transpose(1, 2)
            print(f"logits shape: {logits.shape}")
            print(f"ground_truth_subtrajectory_id shape: {ground_truth_subtrajectory_id.shape}")
            loss = torch.nn.CrossEntropyLoss()(logits, ground_truth_subtrajectory_id)
            # print("HELLO 3")

        
            # CrossEntropyLoss attend :
            # - logits : (N, C)
            # - targets : (N)
        metrics.update(
            {
                "loss_value": loss.item(),  # Detached value for logging
            }
        )

        # Get detailed L1 losses for logging
        should_log_l1_loss = not use_diffusion or (use_diffusion and compute_diffusion_l1)

        if should_log_l1_loss and not use_subtrajectory:
            ground_truth_curr_action = ground_truth_actions[:, 0]
            predicted_curr_action = predicted_actions[:, 0]
            ground_truth_next_actions = ground_truth_actions[:, 1:]
            predicted_next_actions = predicted_actions[:, 1:]
            curr_action_l1_loss = torch.nn.L1Loss()(ground_truth_curr_action, predicted_curr_action)
            next_actions_l1_loss = torch.nn.L1Loss()(ground_truth_next_actions, predicted_next_actions)
            metrics.update(
                {
                    "curr_action_l1_loss": curr_action_l1_loss.item(),
                    "next_actions_l1_loss": next_actions_l1_loss.item(),
                }
            )

    # Return both the loss tensor (with gradients) and the metrics dictionary (with detached values)
    return loss, metrics

def save_training_checkpoint(
    run_dir,
    log_step,
    vla,
    processor,
    proprio_projector,
    noisy_action_projector,
    action_head,
    train_dataset,
    distributed_state,
) -> None:
    """
    Save all training checkpoints including model components, LoRA adapter, and dataset statistics.

    Args:
        cfg (FinetuneConfig): Training configuration.
        run_dir (Path): Experiment run directory path.
        log_step (int): Current logging step.
        vla (OpenVLAForActionPrediction): Vision-language-action policy.
        processor (PrismaticProcessor): OpenVLA inputs processor.
        proprio_projector (nn.Module): Proprioceptive state projector module.
        noisy_action_projector (nn.Module): Noisy action projector module (only used for diffusion).
        action_head (nn.Module): Action head module.
        train_dataset (RLDSDataset): Training dataset.
        distributed_state (PartialState): Distributed training state.

    Returns:
        None.
    """
    # Determine checkpoint paths and naming
    if save_latest_checkpoint_only:
        checkpoint_dir = run_dir
        checkpoint_name_suffix = "latest_checkpoint.pt"
    else:
        checkpoint_dir = Path(str(run_dir) + f"--{log_step}_chkpt")
        checkpoint_name_suffix = f"{log_step}_checkpoint.pt"

    adapter_dir = checkpoint_dir / "lora_adapter"

    # Create directories and save dataset statistics (main process only)
    if distributed_state.is_main_process:
        os.makedirs(checkpoint_dir, exist_ok=True)
        os.makedirs(adapter_dir, exist_ok=True)
        save_dataset_statistics(train_dataset.dataset_statistics, checkpoint_dir)
        print(f"Saving Model Checkpoint for Step {log_step}")

    # Wait for directories to be created
    # dist.barrier()

    # Save model components (main process only)
    if distributed_state.is_main_process:
        # Save processor and LoRA adapter
        processor.save_pretrained(checkpoint_dir)
        vla.save_pretrained(adapter_dir)

        # Save other components
        if use_proprio and proprio_projector is not None:
            torch.save(proprio_projector.state_dict(), checkpoint_dir / f"proprio_projector--{checkpoint_name_suffix}")

        if use_diffusion and noisy_action_projector is not None:
            torch.save(
                noisy_action_projector.state_dict(), checkpoint_dir / f"noisy_action_projector--{checkpoint_name_suffix}"
            )

        if (use_l1_regression or use_diffusion or use_subtrajectory) and action_head is not None:
            torch.save(action_head.state_dict(), checkpoint_dir / f"action_head--{checkpoint_name_suffix}")

        if use_film:
            # To be safe, just save the entire vision backbone (not just FiLM components)
            torch.save(
                vla.vision_backbone.state_dict(), checkpoint_dir / f"vision_backbone--{checkpoint_name_suffix}"
            )

    # Wait for model components to be saved
    # dist.barrier()

    # Merge LoRA weights into base model and save resulting model checkpoint
    # Note: Can be very slow on some devices; if so, we recommend merging offline
    if use_lora and merge_lora_during_training:
        base_vla = AutoModelForVision2Seq.from_pretrained(
            vla_path, torch_dtype=torch.bfloat16, low_cpu_mem_usage=True, trust_remote_code=True
        )
        merged_vla = PeftModel.from_pretrained(base_vla, adapter_dir)
        merged_vla = merged_vla.merge_and_unload()

        if distributed_state.is_main_process:
            merged_vla.save_pretrained(checkpoint_dir)
            print(f"Saved merged model for Step {log_step} at: {checkpoint_dir}")

        # Wait for merged model to be saved
        # dist.barrier()

def run_validation(
    vla,
    action_head,
    noisy_action_projector,
    proprio_projector,
    val_dataloader,
    action_tokenizer,
    device_id,
    num_patches,
    log_step,
    distributed_state,
    val_time_limit,
) -> None:
    """
    Compute validation set metrics for logging.

    Args:
        vla (OpenVLAForActionPrediction): Vision-language-action policy.
        action_head (nn.Module): Action head module.
        noisy_action_projector (nn.Module): Noisy action projector module (only used for diffusion).
        proprio_projector (nn.Module): Proprioceptive state projector module.
        val_dataloader (DataLoader): Validation data loader.
        action_tokenizer (ActionTokenizer): Action tokenizer.
        device_id (str): Device ID.
        cfg (FinetuneConfig): Training configuration.
        num_patches (int): Number of vision patches.
        log_step (int): Current logging step.
        distributed_state (PartialState): Distributed training state.
        val_time_limit (int): Time limit for computing validation metrics.

    Returns:
        None.
    """
    val_start_time = time.time()
    vla.eval()
    val_batches_count = 0

    # List to store validation metrics
    all_val_metrics = []

    with torch.no_grad():
        for batch in val_dataloader:
            # Always compute L1 loss for validation, even for diffusion
            _, metrics = run_forward_pass(
                vla=vla,
                action_head=action_head,
                noisy_action_projector=noisy_action_projector,
                proprio_projector=proprio_projector,
                batch=batch,
                action_tokenizer=action_tokenizer,
                device_id=device_id,
                use_l1_regression=use_l1_regression,
                use_diffusion=use_diffusion,
                use_proprio=use_proprio,
                use_film=use_film,
                num_patches=num_patches,
                compute_diffusion_l1=True,
                num_diffusion_steps_train=num_diffusion_steps_train if use_diffusion else None,
            )
 
            # Add the loss value to the metrics
            metrics["loss"] = metrics["loss_value"]
            all_val_metrics.append(metrics)
            val_batches_count += 1

            # Cut testing on validation set short if it exceeds time limit
            if time.time() - val_start_time > val_time_limit:
                break

    # Compute average validation metrics
    avg_val_metrics = {}
    for metric_name in all_val_metrics[0].keys():
        values = [metrics[metric_name] for metrics in all_val_metrics if metric_name in metrics]
        if values:
            avg_val_metrics[metric_name] = sum(values) / len(values)

    # Add batch count to metrics
    avg_val_metrics["val_batches_count"] = val_batches_count

    # Log validation metrics to W&B
    if distributed_state.is_main_process:
        log_metrics_to_wandb(avg_val_metrics, "VLA Val", log_step, wandb)

Current working directory: /home/ids/ext-5219/tokenizer/openvla-oft


/home/ids/ext-5219/miniconda3/envs/openvla-oft/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using PUSHT constants:
  NUM_ACTIONS_CHUNK = 3
  ACTION_DIM = 1
  PROPRIO_DIM = 8
  ACTION_PROPRIO_NORMALIZATION_TYPE = bounds_q99
If needed, manually set the correct constants in `prismatic/vla/constants.py`!


2026-01-22 17:04:31.634671: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-01-22 17:04:31.634794: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-01-22 17:04:31.716578: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-01-22 17:04:31.871351: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-01-22 17:04:33.708566: W tensorflow/compiler/tf2

Fine-tuning OpenVLA Model `openvla/openvla-7b` 


In [ ]:
# FINE-TUNNING SCRIPT FOR OPENVLA OFT MODEL PART 2 
# PARAMETERS FOR MODEL 


from huggingface_hub import HfApi, snapshot_download 

from prismatic.extern.hf.configuration_prismatic import OpenVLAConfig
from prismatic.extern.hf.processing_prismatic import PrismaticImageProcessor, PrismaticProcessor
from prismatic.extern.hf.modeling_prismatic import OpenVLAForActionPrediction

import wandb
from prismatic.models.film_vit_wrapper import FiLMedPrismaticVisionBackbone
from prismatic.models.projectors import (
    NoisyActionProjector,
    ProprioProjector,
)
from prismatic.vla.constants import (
    ACTION_DIM,
    ACTION_PROPRIO_NORMALIZATION_TYPE,
    NUM_ACTIONS_CHUNK,
    PROPRIO_DIM
)
from prismatic.models.action_heads import DiffusionActionHead, L1RegressionActionHead,SubTrajectoryHead
from torch.optim import AdamW
from torch.optim.lr_scheduler import MultiStepLR
from prismatic.vla.action_tokenizer import ActionTokenizer
from prismatic.vla.datasets import RLDSBatchTransform, RLDSDataset
from prismatic.models.backbones.llm.prompting import PurePromptBuilder
from accelerate import PartialState
from experiments.robot.openvla_utils import (
    check_model_logic_mismatch,
    model_is_on_hf_hub,
    update_auto_map,
)
from prismatic.vla.datasets.rlds.utils.data_utils import save_dataset_statistics
from torch.utils.data import DataLoader
from prismatic.util.data_utils import PaddedCollatorForActionPrediction
from torch.utils.data import DataLoader
from collections import deque
import tqdm
import shutil



# Logging
wandb_entity: str = "pollen"          # Name of WandB entity
wandb_project: str = "openvla_oft"        # Name of WandB project
wandb_log_freq: int = 10                         # WandB logging frequency in steps
run_id=1


# Create experiment run directory
run_dir = run_root_dir / str(run_id)
os.makedirs(run_dir, exist_ok=True)




device_id=0 #I will use only one GPU for the momment



 # GPU setup
distributed_state = PartialState()
device_id = distributed_state.local_process_index
torch.cuda.set_device(device_id)
torch.cuda.empty_cache()

# Initialize wandb logging
if distributed_state.is_main_process:
    wandb.init(entity=wandb_entity, project=wandb_project, name=f"ft+{run_id}")


# Two options:
# (1) Base model is on Hugging Face Hub
#   - Then download it and record the path to the download directory
# (2) Base model is stored locally
#   - Then register model config in HF Auto Classes
# In both cases, we want to check whether any changes have been made to
# the `modeling_prismatic.py` file in this codebase; if so, we will copy
# the file to the downloaded or locally stored checkpoint directory so
# that the user's changes to the VLA class logic go into effect
if model_is_on_hf_hub(vla_path):
    # Download model directly from Hugging Face Hub
    vla_download_path = snapshot_download(repo_id=vla_path)
    # Overwrite VLA path
    vla_path = vla_download_path
else:
    # Register OpenVLA model to HF Auto Classes (not needed if the model is on HF Hub)
    AutoConfig.register("openvla", OpenVLAConfig)
    AutoImageProcessor.register(OpenVLAConfig, PrismaticImageProcessor)
    AutoProcessor.register(OpenVLAConfig, PrismaticProcessor)
    AutoModelForVision2Seq.register(OpenVLAConfig, OpenVLAForActionPrediction)
 


# Update config.json and sync model files
if distributed_state.is_main_process:
    update_auto_map(vla_path)
    check_model_logic_mismatch(vla_path)

# Wait for model files to be synced
# dist.barrier()



# Load processor and VLA
processor = AutoProcessor.from_pretrained(vla_path, trust_remote_code=True)
vla = AutoModelForVision2Seq.from_pretrained(
    vla_path,
    torch_dtype=torch.bfloat16,
    low_cpu_mem_usage=True,
    trust_remote_code=True,
).to(device_id)

# Set number of images in VLA input
vla.vision_backbone.set_num_images_in_input(num_images_in_input)

# LoRA setup
if use_lora:
    lora_config = LoraConfig(
        r=lora_rank,
        lora_alpha=min(lora_rank, 16),
        lora_dropout=lora_dropout,
        target_modules="all-linear",
        init_lora_weights="gaussian",
    )
    vla = get_peft_model(vla, lora_config)
    vla.print_trainable_parameters()

# FiLM setup
if use_film:
    # Wrap vision backbone with FiLM wrapper
    # Important: For this, must specify `vla.model.vision_backbone` instead of just `vla.vision_backbone`, since the
    # latter would cause the new wrapped backbone to be saved as a new attribute of `vla` instead of overwriting the
    # original one (due to the LoRA wrapper)
    vla.model.vision_backbone = FiLMedPrismaticVisionBackbone(
        vision_backbone=vla.model.vision_backbone,
        llm_dim=vla.llm_dim,
    )
    if resume:
        state_dict = load_checkpoint("vision_backbone", vla_path,resume_step)
        vla.model.vision_backbone.load_state_dict(state_dict)
    vla.model.vision_backbone = vla.model.vision_backbone.to(device_id)

# Wrap VLA with DDP
# vla = wrap_ddp(vla, device_id, find_unused=True)
action_head=None
# If applicable, instantiate proprio projector
if use_proprio:
    proprio_projector = init_module(
        ProprioProjector,
        "proprio_projector",
        device_id,
        {"llm_dim": vla.llm_dim, "proprio_dim": PROPRIO_DIM},
    )

# If applicable, instantiate continuous action head for L1 regression
if use_l1_regression:
    action_head = init_module(
        L1RegressionActionHead,
        "action_head",
        device_id,
        {"input_dim": vla.llm_dim, "hidden_dim": vla.llm_dim, "action_dim": ACTION_DIM},
        to_bf16=True,
    )

if use_subtrajectory:
    # action_head = init_module(
    #     SubTrajectoryHead,
    #     "action_head",
    #     device_id,
    #     {"input_dim": vla.llm_dim, "hidden_dim": vla.llm_dim, "num_subtrajectory_ids": SUBTRAJECTORY_DIM},
    #     to_bf16=True,
    # )
    action_head = SubTrajectoryHead(
            input_dim=vla.llm_dim,
            hidden_dim=vla.llm_dim,
            num_subtrajectory_ids=SUBTRAJECTORY_DIM,
        ).to(device_id)

# If applicable, instantiate diffusion action head and noisy action projector
if use_diffusion:
    action_head = init_module(
        DiffusionActionHead,
        "action_head",
        device_id,
        {
            "input_dim": vla.llm_dim,
            "hidden_dim": vla.llm_dim,
            "action_dim": ACTION_DIM,
            "num_diffusion_steps_train": num_diffusion_steps_train,
        },
        to_bf16=True,
    )
    noisy_action_projector = init_module(
        NoisyActionProjector, "noisy_action_projector", device_id, {"llm_dim": vla.llm_dim}
    )

# Get number of vision patches
NUM_PATCHES = vla.vision_backbone.get_num_patches() * vla.vision_backbone.get_num_images_in_input()
# If we have proprio inputs, a single proprio embedding is appended to the end of the vision patch embeddings
if use_proprio:
    NUM_PATCHES += 1
# For diffusion, a single diffusion timestep embedding is appended to the end of the vision patch embeddings
if use_diffusion:
    NUM_PATCHES += 1

# Instantiate optimizer
trainable_params = [param for param in vla.parameters() if param.requires_grad]
if use_l1_regression or use_diffusion or use_subtrajectory:
    trainable_params += [param for param in action_head.parameters() if param.requires_grad]
if use_diffusion:
    trainable_params += [param for param in noisy_action_projector.parameters() if param.requires_grad]
if use_proprio:
    trainable_params += [param for param in proprio_projector.parameters() if param.requires_grad]
print(f"# total trainable params: {sum(p.numel() for p in trainable_params)}")
optimizer = AdamW(trainable_params, lr=learning_rate)

# Record original learning rate
original_lr = optimizer.param_groups[0]["lr"]

# Create learning rate scheduler
scheduler = MultiStepLR(
    optimizer,
    milestones=[num_steps_before_decay],  # Number of steps after which LR will change
    gamma=0.1,  # Multiplicative factor of learning rate decay
)

# Create Action Tokenizer
action_tokenizer = ActionTokenizer(processor.tokenizer)

# Load Fine-tuning Dataset =>> note that we use an RLDS-formatted dataset following Open X-Embodiment by default.
#   =>> If you want to use a non-RLDS dataset (e.g., a standard PyTorch Dataset) see the following commented block.
#   =>> Note that our training code does not loop over epochs because the RLDS loader does this implicitly; if using
#       your own Dataset, make sure to add the appropriate logic to the training loop!
#
# ---
# from prismatic.vla.datasets import DummyDataset
#
# train_dataset = DummyDataset(
#     action_tokenizer,
#     processor.tokenizer,
#     image_transform=processor.image_processor.apply_transform,
#     prompt_builder_fn=PurePromptBuilder,
# )
# ---

# We assume that the model takes as input one third-person camera image and 1 or 2 optional wrist camera image(s)
use_wrist_image = num_images_in_input > 1



wandb: Currently logged in as: cataclysme-apocalypse (pollen) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Fetching 18 files: 100%|█|


Created backup of original config at: /home/ids/ext-5219/.cache/huggingface/hub/models--openvla--openvla-7b/snapshots/31f090d05236101ebfc381b61c674dd4746d4ce0/config.json.back.20260122_170448
Updated config.json at: /home/ids/ext-5219/.cache/huggingface/hub/models--openvla--openvla-7b/snapshots/31f090d05236101ebfc381b61c674dd4746d4ce0/config.json
Changes made:
  - Set AutoConfig to "configuration_prismatic.OpenVLAConfig"
  - Set AutoModelForVision2Seq to "modeling_prismatic.OpenVLAForActionPrediction"


Loading checkpoint shards:


trainable params: 110,828,288 || all params: 7,652,065,472 || trainable%: 1.4483
# total trainable params: 161327902


In [ ]:
# FINE-TUNNING SCRIPT FOR OPENVLA OFT MODEL PART 3
# LOADING DATASET
# Create training and optional validation datasets


batch_transform = RLDSBatchTransform(
    action_tokenizer,
    processor.tokenizer,
    image_transform=processor.image_processor.apply_transform,
    prompt_builder_fn=PurePromptBuilder,
    use_wrist_image=use_wrist_image,
    use_proprio=use_proprio,
    use_subtrajectory=use_subtrajectory,
)
train_dataset = RLDSDataset(
    data_root_dir,
    dataset_name,
    batch_transform,
    resize_resolution=tuple(vla.config.image_sizes),
    shuffle_buffer_size=shuffle_buffer_size,
    image_aug=image_aug,
)

if use_val_set:
    val_dataset = RLDSDataset(
        data_root_dir,
        dataset_name,
        batch_transform,
        resize_resolution=tuple(vla.config.image_sizes),
        shuffle_buffer_size=shuffle_buffer_size // 10,
        image_aug=image_aug,
        train=False,
    )

# [Important] Save dataset statistics so that we can unnormalize actions during inference
if distributed_state.is_main_process:
    save_dataset_statistics(train_dataset.dataset_statistics, run_dir)

# Create collator and dataloader
collator = PaddedCollatorForActionPrediction(
    processor.tokenizer.model_max_length, processor.tokenizer.pad_token_id, padding_side="right"
)
dataloader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    sampler=None,
    collate_fn=collator,
    num_workers=0,  # Important: Set to 0 if using RLDS, which uses its own parallelism
)
if use_val_set:
    val_batch_size = batch_size
    val_dataloader = DataLoader(
        val_dataset,
        batch_size=val_batch_size,
        sampler=None,
        collate_fn=collator,
        num_workers=0,  # Important: Set to 0 if using RLDS, which uses its own parallelism
    )



01/22 [17:05:28] INFO     | >> Load dataset info from                                           ]8;id=81003;file:///home/ids/ext-5219/miniconda3/envs/openvla-oft/lib/python3.10/site-packages/tensorflow_datasets/core/dataset_info.py\dataset_info.py]8;;\:]8;id=14569;file:///home/ids/ext-5219/miniconda3/envs/openvla-oft/lib/python3.10/site-packages/tensorflow_datasets/core/dataset_info.py#599\599]8;;\
                          /home/ids/ext-5219/tokenizer/test/columbia_cairlab_pusht_real/1.0.0                      

                 WARNING  | >> `FeatureConnector.dtype` is deprecated. Please change your code to use ]8;id=358848;file:///home/ids/ext-5219/miniconda3/envs/openvla-oft/lib/python3.10/site-packages/tensorflow_datasets/core/features/feature.py\feature.py]8;;\:]8;id=449437;file:///home/ids/ext-5219/miniconda3/envs/openvla-oft/lib/python3.10/site-packages/tensorflow_datasets/core/features/feature.py#67\67]8;;\
                          NumPy with the field `FeatureConnector.np_dtype` or use TensorFlow with the              
                          field `FeatureConnector.tf_dtype`.                                                       

                 WARNING  | >> `FeatureConnector.dtype` is deprecated. Please change your code to use ]8;id=724811;file:///home/ids/ext-5219/miniconda3/envs/openvla-oft/lib/python3.10/site-packages/tensorflow_datasets/core/features/feature.py\feature.py]8;;\:]8;id=93894;file:///home/ids/ext-5219/miniconda3/envs/openvla-oft/lib/python3.10/site-packages/tensorflow_datasets/core/features/feature.py#67\67]8;;\
                          NumPy with the field `FeatureConnector.np_dtype` or use TensorFlow with the              
                          field `FeatureConnector.tf_dtype`.                                                       

                 INFO     | >> Constructing tf.data.Dataset columbia_cairlab_pusht_real for    ]8;id=480254;file:///home/ids/ext-5219/miniconda3/envs/openvla-oft/lib/python3.10/site-packages/tensorflow_datasets/core/logging/logging_logger.py\logging_logger.py]8;;\:]8;id=596399;file:///home/ids/ext-5219/miniconda3/envs/openvla-oft/lib/python3.10/site-packages/tensorflow_datasets/core/logging/logging_logger.py#49\49]8;;\
                          split all, from                                                                          
                          /home/ids/ext-5219/tokenizer/test/columbia_cairlab_pusht_real/1.0.0                      

2026-01-22 17:05:28.858918: I tensorflow/core/grappler/optimizers/data/replicate_on_split.cc:32] Running replicate on split optimization


...

                 WARNING  | >> `FeatureConnector.dtype` is deprecated. Please change your code to use ]8;id=115031;file:///home/ids/ext-5219/miniconda3/envs/openvla-oft/lib/python3.10/site-packages/tensorflow_datasets/core/features/feature.py\feature.py]8;;\:]8;id=187078;file:///home/ids/ext-5219/miniconda3/envs/openvla-oft/lib/python3.10/site-packages/tensorflow_datasets/core/features/feature.py#67\67]8;;\
                          NumPy with the field `FeatureConnector.np_dtype` or use TensorFlow with the              
                          field `FeatureConnector.tf_dtype`.                                                       

                 INFO     | >> Constructing tf.data.Dataset columbia_cairlab_pusht_real for    ]8;id=244335;file:///home/ids/ext-5219/miniconda3/envs/openvla-oft/lib/python3.10/site-packages/tensorflow_datasets/core/logging/logging_logger.py\logging_logger.py]8;;\:]8;id=32179;file:///home/ids/ext-5219/miniconda3/envs/openvla-oft/lib/python3.10/site-packages/tensorflow_datasets/core/logging/logging_logger.py#49\49]8;;\
                          split train, from                                                                        
                          /home/ids/ext-5219/tokenizer/test/columbia_cairlab_pusht_real/1.0.0                      

2026-01-22 17:05:29.895410: I tensorflow/core/grappler/optimizers/data/replicate_on_split.cc:32] Running replicate on split optimization


01/22 [17:05:30] INFO     | >> [*] Applying frame transforms on dataset...                           ]8;id=566306;file:///home/ids/ext-5219/tokenizer/openvla-oft/prismatic/vla/datasets/rlds/dataset.py\dataset.py]8;;\:]8;id=433362;file:///home/ids/ext-5219/tokenizer/openvla-oft/prismatic/vla/datasets/rlds/dataset.py#582\582]8;;\

01/22 [17:05:32] INFO     | >> [*] Saved dataset statistics file at path                          ]8;id=295283;file:///home/ids/ext-5219/tokenizer/openvla-oft/prismatic/vla/datasets/rlds/utils/data_utils.py\data_utils.py]8;;\:]8;id=858578;file:///home/ids/ext-5219/tokenizer/openvla-oft/prismatic/vla/datasets/rlds/utils/data_utils.py#284\284]8;;\
                          runs/1/dataset_statistics.json                                                           

In [ ]:
# FINE-TUNNING SCRIPT FOR OPENVLA OFT MODEL PART 4
#TRAINING LOOP  # ==================================================
# 
# 
# 
#  Deque to store recent train metrics (used for computing smoothened metrics for gradient accumulation)
recent_metrics = {
    "loss_value": deque(maxlen=grad_accumulation_steps),
    "curr_action_accuracy": deque(maxlen=grad_accumulation_steps),
    "curr_action_l1_loss": deque(maxlen=grad_accumulation_steps),
    "next_actions_accuracy": deque(maxlen=grad_accumulation_steps),
    "next_actions_l1_loss": deque(maxlen=grad_accumulation_steps),
}

# Start training
with tqdm.tqdm(total=max_steps, leave=False) as progress:
    vla.train()
    optimizer.zero_grad()
    for batch_idx, batch in enumerate(dataloader):
        # Compute training metrics and loss
        compute_diffusion_l1 = use_diffusion and batch_idx % diffusion_sample_freq == 0
        loss, metrics = run_forward_pass(
            vla=vla,
            action_head=action_head,
            noisy_action_projector=noisy_action_projector if use_diffusion else None,
            proprio_projector=proprio_projector if use_proprio else None,
            batch=batch,
            action_tokenizer=action_tokenizer,
            device_id=device_id,
            use_l1_regression=use_l1_regression,
            use_diffusion=use_diffusion,
            use_proprio=use_proprio,
            use_film=use_film,
            num_patches=NUM_PATCHES,
            compute_diffusion_l1=compute_diffusion_l1,
            num_diffusion_steps_train=num_diffusion_steps_train if use_diffusion else None,
            use_subtrajectory=use_subtrajectory,
        )

        # Normalize loss to account for gradient accumulation
        normalized_loss = loss / grad_accumulation_steps

        # Backward pass
        normalized_loss.backward()

        # Store recent train metrics
        for metric_name, value in metrics.items():
            if metric_name in recent_metrics:
                recent_metrics[metric_name].append(value)

        # Compute gradient step index
        gradient_step_idx = batch_idx // grad_accumulation_steps

        # Compute smoothened train metrics
        smoothened_metrics = compute_smoothened_metrics(recent_metrics)

        # Push Metrics to W&B (every wandb_log_freq gradient steps)
        log_step = gradient_step_idx if not resume else resume_step + gradient_step_idx
        if distributed_state.is_main_process and log_step % wandb_log_freq == 0:
            log_metrics_to_wandb(smoothened_metrics, "VLA Train", log_step, wandb)

        # [If applicable] Linearly warm up learning rate from 10% to 100% of original
        if lr_warmup_steps > 0:
            lr_progress = min((gradient_step_idx + 1) / lr_warmup_steps, 1.0)  # Cap at 1.0
            current_lr = original_lr * (0.1 + 0.9 * lr_progress)
            for param_group in optimizer.param_groups:
                param_group["lr"] = current_lr

        if distributed_state.is_main_process and gradient_step_idx % wandb_log_freq == 0:
            # Log the learning rate
            # Make sure to do this AFTER any learning rate modifications (e.g., warmup/decay)
            wandb.log(
                {
                    "VLA Train/Learning Rate": scheduler.get_last_lr()[0],
                },
                step=log_step,
            )

        # Optimizer and LR scheduler step
        if (batch_idx + 1) % grad_accumulation_steps == 0:
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
            progress.update()

        # Save model checkpoint: either keep latest checkpoint only or all checkpoints
        if gradient_step_idx > 0 and log_step % save_freq == 0:
            save_training_checkpoint(
                run_dir=run_dir,
                log_step=log_step,
                vla=vla,
                processor=processor,
                proprio_projector=proprio_projector if use_proprio else None,
                noisy_action_projector=noisy_action_projector if use_diffusion else None,
                action_head=action_head if (use_l1_regression or use_diffusion or use_subtrajectory) else None,
                train_dataset=train_dataset,
                distributed_state=distributed_state,
            )

        # Test model on validation set
        if use_val_set and log_step > 0 and log_step % val_freq == 0:
            run_validation(
                vla=vla,
                action_head=action_head,
                noisy_action_projector=noisy_action_projector if use_diffusion else None,
                proprio_projector=proprio_projector if use_proprio else None,
                val_dataloader=val_dataloader,
                action_tokenizer=action_tokenizer,
                device_id=device_id,
                num_patches=NUM_PATCHES,
                log_step=log_step,
                distributed_state=distributed_state,
                val_time_limit=val_time_limit,
            )
            # Set model back to training mode after validation
            vla.train()

        # Stop training when max_steps is reached
        if log_step == max_steps:
            print(f"Max step {max_steps} reached! Stopping training...")
            break

  0%| | 0/300 [00:00<?, ?iWARNING: All log messages before absl::InitializeLog() is called are written to STDERR
W0000 00:00:1769097932.524782 2893181 op_level_cost_estimator.cc:699] Error in PredictCost() for the op: op: "CropAndResize" attr { key: "T" value { type: DT_FLOAT } } attr { key: "extrapolation_value" value { f: 0 } } attr { key: "method" value { s: "bilinear" } } inputs { dtype: DT_FLOAT shape { dim { size: 1 } dim { size: 224 } dim { size: 224 } dim { size: 3 } } } inputs { dtype: DT_FLOAT shape { dim { size: -2 } dim { size: 4 } } } inputs { dtype: DT_INT32 shape { dim { size: -2 } } } inputs { dtype: DT_INT32 shape { dim { size: 2 } } } device { type: "CPU" vendor: "AuthenticAMD" model: "241" frequency: 2994 num_cores: 8 environment { key: "cpu_instruction_set" value: "AVX SSE, SSE2, SSE3, SSSE3, SSE4.1, SSE4.2" } environment { key: "eigen" value: "3.4.90" } l1_cache_size: 32768 l2_cache_size: 524288 l3_cache_size: 134217728 memory_size: 268435456 } outputs { dtype: DT_

--- DEBUG: traj_len 246 ---
--- DEBUG: future_action_window_size 2 ---
--- DEBUG: traj_CLUSTERID_AVANT [246] ---
--- DEBUG: traj_ACTION [244 3 7] ---
--- DEBUG: traj_CLUSTERID_APRES [244 3] ---
--- DEBUG: traj_len 136 ---
--- DEBUG: future_action_window_size 2 ---
--- DEBUG: traj_CLUSTERID_AVANT [136] ---
--- DEBUG: traj_ACTION [134 3 7] ---
--- DEBUG: traj_CLUSTERID_APRES [134 3] ---
--- DEBUG: traj_len 217 ---
--- DEBUG: future_action_window_size 2 ---
--- DEBUG: traj_CLUSTERID_AVANT [217] ---
--- DEBUG: traj_ACTION [215 3 7] ---
--- DEBUG: traj_CLUSTERID_APRES [215 3] ---
--- DEBUG: traj_len 237 ---
--- DEBUG: future_action_window_size 2 ---
--- DEBUG: traj_CLUSTERID_AVANT [237] ---
--- DEBUG: traj_ACTION [235 3 7] ---
--- DEBUG: traj_CLUSTERID_APRES [235 3] ---
--- DEBUG: traj_len 179 ---
--- DEBUG: future_action_window_size 2 ---
--- DEBUG: traj_CLUSTERID_AVANT [179] ---
--- DEBUG: traj_ACTION [177 3 7] ---
--- DEBUG: traj_CLUSTERID_APRES [177 3] ---
--- DEBUG: traj_len 184 ---
--

  0%| | 1/300 [00:06<33:03

--- DEBUG: action shape (3, 7) ---
--- DEBUG: future action shape (2, 7) ---
--- DEBUG: string current action shape 7 ---
--- DEBUG: string future action shape 14 ---
--- DEBUG: string  action shape 3 ---
dimension cluster: (3,)
--- DEBUG: action shape (3, 7) ---
--- DEBUG: future action shape (2, 7) ---
--- DEBUG: string current action shape 7 ---
--- DEBUG: string future action shape 14 ---
--- DEBUG: string  action shape 3 ---
dimension cluster: (3,)
--- DEBUG: action shape (3, 7) ---
--- DEBUG: future action shape (2, 7) ---
--- DEBUG: string current action shape 7 ---
--- DEBUG: string future action shape 14 ---
--- DEBUG: string  action shape 3 ---
dimension cluster: (3,)
--- DEBUG: action shape (3, 7) ---
--- DEBUG: future action shape (2, 7) ---
--- DEBUG: string current action shape 7 ---
--- DEBUG: string future action shape 14 ---
--- DEBUG: string  action shape 3 ---
dimension cluster: (3,)
--- DEBUG: action shape (3, 7) ---
--- DEBUG: future action shape (2, 7) ---
--- DEB

  1%| | 2/300 [00:07<15:44

...

301it [16:16,  1.30it/s]  

01/22 [17:21:48] INFO     | >> [*] Saved dataset statistics file at path                          ]8;id=518919;file:///home/ids/ext-5219/tokenizer/openvla-oft/prismatic/vla/datasets/rlds/utils/data_utils.py\data_utils.py]8;;\:]8;id=616768;file:///home/ids/ext-5219/tokenizer/openvla-oft/prismatic/vla/datasets/rlds/utils/data_utils.py#284\284]8;;\
                          runs/1/dataset_statistics.json                                                           

Saving Model Checkpoint for Step 300


/home/ids/ext-5219/miniconda3/envs/openvla-oft/lib/python3.10/site-packages/peft/utils/save_and_load.py:180: UserWarning: Setting `save_embedding_layers` to `True` as embedding layers found in `target_modules`.
  warnings.warn("Setting `save_embedding_layers` to `True` as embedding layers found in `target_modules`.")
Loading checkpoint shards:


Saved merged model for Step 300 at: runs/1
Max step 300 reached! Stopping training...


# INFERENCE OPENVLA

In [6]:
from dataclasses import dataclass
from typing import Optional, Union
from pathlib import Path


@dataclass
class GenerateConfig:
    # fmt: off

    #################################################################################################################
    # Model-specific parameters
    #################################################################################################################
    model_family: str = "openvla"                    # Model family
    pretrained_checkpoint: Union[str, Path] = ""     # Pretrained checkpoint path

    use_l1_regression: bool = True                   # If True, uses continuous action head with L1 regression objective
    use_diffusion: bool = False                      # If True, uses continuous action head with diffusion modeling objective (DDIM)
    num_diffusion_steps_train: int = 50              # (When `diffusion==True`) Number of diffusion steps used for training
    num_diffusion_steps_inference: int = 50          # (When `diffusion==True`) Number of diffusion steps used for inference
    use_film: bool = False                           # If True, uses FiLM to infuse language inputs into visual features
    num_images_in_input: int = 2                     # Number of images in the VLA input (default: 1)
    use_proprio: bool = True                         # Whether to include proprio state in input
    use_subtrajectory: bool = True 
    SUBTRAJECTORY_DIM: int = 30

    center_crop: bool = True                         # Center crop? (if trained w/ random crop image aug)
    num_open_loop_steps: int = 8                     # Number of actions to execute open-loop before requerying policy

    lora_rank: int = 32                              # Rank of LoRA weight matrix (MAKE SURE THIS MATCHES TRAINING!)

    unnorm_key: Union[str, Path] =  ""            # Action un-normalization key

    load_in_8bit: bool = False                       # (For OpenVLA only) Load with 8-bit quantization
    load_in_4bit: bool = False                       # (For OpenVLA only) Load with 4-bit quantization

    #################################################################################################################
    # Utils
    #################################################################################################################
    run_id_note: Optional[str] = None                # Extra note to add to end of run ID for logging
    local_log_dir: str = "./experiments/logs"        # Local directory for eval logs

    use_wandb: bool = False                          # Whether to also log results in Weights & Biases
    wandb_entity: str = "your-wandb-entity"          # Name of WandB entity
    wandb_project: str = "your-wandb-project"        # Name of WandB project

    seed: int = 7                                    # Random Seed (for reproducibility)

    # fmt: on

In [7]:
import sys
import os

os.chdir("/home/ids/ext-5219/tokenizer/openvla-oft/")
print("Current working directory:", os.getcwd())

sys.argv.append("pusht")

from prismatic.vla.constants import (
    ACTION_DIM,
    ACTION_PROPRIO_NORMALIZATION_TYPE,
    NUM_ACTIONS_CHUNK,
    PROPRIO_DIM,
)

from PIL import Image
import numpy as np
from pathlib import Path
from prismatic.vla.datasets import RLDSBatchTransform, RLDSDataset
from experiments.robot.openvla_utils import (
    get_action_head, 
    get_processor, 
    get_proprio_projector, 
    get_vla, 
    get_vla_action,
    check_model_logic_mismatch,
    update_auto_map,
)
from prismatic.vla.constants import NUM_ACTIONS_CHUNK, PROPRIO_DIM

# Instantiate config
cfg = GenerateConfig(
    pretrained_checkpoint ="/home/ids/ext-5219/tokenizer/openvla-oft/runs/1/",
    use_l1_regression = False,
    use_diffusion = False,
    use_film = False,
    num_images_in_input = 1,
    use_proprio = False,
    load_in_8bit = False,
    load_in_4bit = False,
    center_crop = True,
    num_open_loop_steps = NUM_ACTIONS_CHUNK,
    unnorm_key = "cluster",
    use_subtrajectory=True,
)

# Sync the corrected modeling_prismatic.py before loading the model
update_auto_map(str(cfg.pretrained_checkpoint))
check_model_logic_mismatch(str(cfg.pretrained_checkpoint))
print("✓ Fichier modeling_prismatic.py synchronisé")

# Load OpenVLA-OFT policy and inputs processor
vla = get_vla(cfg)
processor = get_processor(cfg)
proprio_projector = None
action_head = get_action_head(cfg, llm_dim=vla.llm_dim)

print("✓ Modèle chargé avec succès !")

# Load dataset via RLDSDataset (same as training pipeline)
data_root_dir: Path = Path("/home/ids/ext-5219/tokenizer/test")
dataset_name: str = "columbia_cairlab_pusht_real"

# Create batch transform
action_tokenizer_inf = ActionTokenizer(processor.tokenizer)
batch_transform_inf = RLDSBatchTransform(
    action_tokenizer_inf,
    processor.tokenizer,
    image_transform=processor.image_processor.apply_transform,
    prompt_builder_fn=PurePromptBuilder,
    use_wrist_image=False,
    use_proprio=False,
    use_subtrajectory=True,
)

# Create dataset (this properly handles the data structure)
inference_dataset = RLDSDataset(
    data_root_dir,
    dataset_name,
    batch_transform_inf,
    resize_resolution=tuple(vla.config.image_sizes),
    shuffle_buffer_size=100,
    image_aug=False,
    train=True,
)

# Get first sample
sample_iterator = iter(inference_dataset)
sample_dict = next(sample_iterator)

print(f"✓ Dataset loaded. Sample keys: {sample_dict.keys()}")

# Extract ground truth subtrajectory_id (check if it exists)
has_subtrajectory = "subtrajectory_id" in sample_dict
if has_subtrajectory:
    ground_truth_subtrajectory_id = sample_dict["subtrajectory_id"].item() if hasattr(sample_dict["subtrajectory_id"], "item") else int(sample_dict["subtrajectory_id"])
    print(f"✓ Ground truth subtrajectory_id found: {ground_truth_subtrajectory_id}")
else:
    ground_truth_subtrajectory_id = None
    print("⚠ No subtrajectory_id in dataset")

# Extract and prepare observation for inference
pixel_values = sample_dict["pixel_values"]

# Convert from tensor to numpy if needed
if hasattr(pixel_values, "numpy"):
    pixel_values = pixel_values.numpy()

# The processor may create multi-channel images (e.g., 6 channels for 2 images)
# Extract only the first 3 channels (primary image)
if pixel_values.shape[0] > 3:
    pixel_values = pixel_values[:3]

# Now pixel_values should be (3, H, W) - transpose to (H, W, 3) using numpy
image_np = np.transpose(pixel_values, (1, 2, 0))

# Denormalize from ImageNet normalization to [0, 255]
if image_np.dtype in [np.float32, np.float64]:
    imagenet_mean = np.array([0.485, 0.456, 0.406])
    imagenet_std = np.array([0.229, 0.224, 0.225])
    image_np = (image_np * imagenet_std[np.newaxis, np.newaxis, :]) + imagenet_mean[np.newaxis, np.newaxis, :]
    image_np = np.clip(image_np, 0, 1)
    image_np = (image_np * 255).astype(np.uint8)
else:
    image_np = image_np.astype(np.uint8)

# Build observation for inference
observation_pour_test = {
    "full_image": image_np,
    "task_description": "push the button",
    "state": None
}

print("✓ Observation préparée avec succès !")
print(f"Image shape: {image_np.shape}, dtype: {image_np.dtype}")

# Generate robot action chunk (sequence of future actions)
print("\n→ Starting inference with get_vla_action()...")
actions = get_vla_action(cfg, vla, processor, observation_pour_test, observation_pour_test["task_description"], action_head, proprio_projector)

print("✓ Generated action chunk:")
for act in actions:
    print(act)

# Display results with ground truth comparison
predicted_cluster = int(actions[0]) if len(actions) > 0 else None

print(f"\n{'='*60}")
print(f"INFERENCE RESULTS:")
print(f"{'='*60}")
if has_subtrajectory:
    gt_cluster = int(ground_truth_subtrajectory_id)
    print(f"Ground Truth Cluster ID:     {gt_cluster}")
    print(f"Predicted Cluster ID:        {predicted_cluster}")
    match = "✓ CORRECT" if gt_cluster == predicted_cluster else "✗ MISMATCH"
    print(f"Result:                      {match}")
else:
    print(f"Predicted Cluster ID:        {predicted_cluster}")
    print("(Ground truth not available - using original dataset)")
print(f"{'='*60}")

Current working directory: /home/ids/ext-5219/tokenizer/openvla-oft
Created backup of original config at: /home/ids/ext-5219/tokenizer/openvla-oft/runs/1/config.json.back.20260122_174813
Updated config.json at: /home/ids/ext-5219/tokenizer/openvla-oft/runs/1/config.json
Changes made:
  - Set AutoConfig to "configuration_prismatic.OpenVLAConfig"
  - Set AutoModelForVision2Seq to "modeling_prismatic.OpenVLAForActionPrediction"
✓ Fichier modeling_prismatic.py synchronisé
Instantiating pretrained VLA policy...
Created backup of original config at: /home/ids/ext-5219/tokenizer/openvla-oft/runs/1/config.json.back.20260122_174813
Updated config.json at: /home/ids/ext-5219/tokenizer/openvla-oft/runs/1/config.json
Changes made:
  - Set AutoConfig to "configuration_prismatic.OpenVLAConfig"
  - Set AutoModelForVision2Seq to "modeling_prismatic.OpenVLAForActionPrediction"


Loading checkpoint shards:


✓ Modèle chargé avec succès !


01/22 [17:48:48] INFO     | >> Load dataset info from                                           ]8;id=601078;file:///home/ids/ext-5219/miniconda3/envs/openvla-oft/lib/python3.10/site-packages/tensorflow_datasets/core/dataset_info.py\dataset_info.py]8;;\:]8;id=188210;file:///home/ids/ext-5219/miniconda3/envs/openvla-oft/lib/python3.10/site-packages/tensorflow_datasets/core/dataset_info.py#599\599]8;;\
                          /home/ids/ext-5219/tokenizer/test/columbia_cairlab_pusht_real/1.0.0                      

                 WARNING  | >> `FeatureConnector.dtype` is deprecated. Please change your code to use ]8;id=748084;file:///home/ids/ext-5219/miniconda3/envs/openvla-oft/lib/python3.10/site-packages/tensorflow_datasets/core/features/feature.py\feature.py]8;;\:]8;id=797066;file:///home/ids/ext-5219/miniconda3/envs/openvla-oft/lib/python3.10/site-packages/tensorflow_datasets/core/features/feature.py#67\67]8;;\
                          NumPy with the field `FeatureConnector.np_dtype` or use TensorFlow with the              
                          field `FeatureConnector.tf_dtype`.                                                       

...

                 INFO     | >> Constructing tf.data.Dataset columbia_cairlab_pusht_real for    ]8;id=69400;file:///home/ids/ext-5219/miniconda3/envs/openvla-oft/lib/python3.10/site-packages/tensorflow_datasets/core/logging/logging_logger.py\logging_logger.py]8;;\:]8;id=476366;file:///home/ids/ext-5219/miniconda3/envs/openvla-oft/lib/python3.10/site-packages/tensorflow_datasets/core/logging/logging_logger.py#49\49]8;;\
                          split train, from                                                                        
                          /home/ids/ext-5219/tokenizer/test/columbia_cairlab_pusht_real/1.0.0                      

2026-01-22 17:48:49.400419: I tensorflow/core/grappler/optimizers/data/replicate_on_split.cc:32] Running replicate on split optimization


                 INFO     | >> [*] Applying frame transforms on dataset...                           ]8;id=739143;file:///home/ids/ext-5219/tokenizer/openvla-oft/prismatic/vla/datasets/rlds/dataset.py\dataset.py]8;;\:]8;id=730781;file:///home/ids/ext-5219/tokenizer/openvla-oft/prismatic/vla/datasets/rlds/dataset.py#582\582]8;;\

--- DEBUG: traj_len 195 ---
--- DEBUG: future_action_window_size 2 ---
--- DEBUG: traj_CLUSTERID_AVANT [195] ---
--- DEBUG: traj_ACTION [193 3 7] ---
--- DEBUG: traj_CLUSTERID_APRES [193 3] ---
--- DEBUG: traj_len 129 ---
--- DEBUG: future_action_window_size 2 ---
--- DEBUG: traj_CLUSTERID_AVANT [129] ---
--- DEBUG: traj_ACTION [127 3 7] ---
--- DEBUG: traj_CLUSTERID_APRES [127 3] ---
--- DEBUG: traj_len 166 ---
--- DEBUG: future_action_window_size 2 ---
--- DEBUG: traj_CLUSTERID_AVANT [166] ---
--- DEBUG: traj_ACTION [164 3 7] ---
--- DEBUG: traj_CLUSTERID_APRES [164 3] ---
--- DEBUG: traj_len 271 ---
--- DEBUG: future_action_window_size 2 ---
--- DEBUG: action shape (3, 7) ---
--- DEBUG: future action shape (2, 7) ---
--- DEBUG: string current action shape 7 ---
--- DEBUG: string future action shape 14 ---
--- DEBUG: string  action shape 3 ---
--- DEBUG: traj_CLUSTERID_AVANT [271] ---
--- DEBUG: traj_ACTION [269 3 7] ---
dimension cluster: (3,)
✓ Dataset loaded. Sample keys: dict_key

ValueError: can only convert an array of size 1 to a Python scalar